<div style='width:100%; display:flex; justify-content:space-between; align-items:center; margin: 1em 0;'>
  <a href='../3.%20application_structure/9.%20configuration_management.ipynb' style='padding:6px 14px; background:#007bff; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>← Chapter 9: Configuration</a>
  <a href='../TOC.md' style='padding:6px 14px; background:#6c757d; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>📚 Table of Contents</a>
  <a href='11.%20security_best_practices.ipynb' style='padding:6px 14px; background:#28a745; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>Chapter 11: Security →</a>
</div>

# Chapter 10: User Authentication with Flask-Login — Who Are You?

> **Your blog needs user accounts. Without authentication, anyone can post, edit, or delete anything.**
> Authentication answers *"Who are you?"* Authorization answers *"What are you allowed to do?"*
> This chapter tackles the first question completely.

By the end of this chapter you will be able to:
- Set up `Flask-Login` with a `UserMixin` model and a `user_loader` callback
- Hash and verify passwords with Werkzeug's `generate_password_hash` / `check_password_hash`
- Build complete register and login routes
- Protect views with `@login_required` and redirect unauthenticated users gracefully
- Access `current_user` in both Python views and Jinja2 templates
- Handle edge cases: missing decorator, deleted users, "Remember Me" cookies

> ❓ **What is a decorator?** A decorator is a function that wraps another function to add behaviour before or after it runs. `@app.route('/')` is shorthand for `index = app.route('/')(index)` — it registers your view function with Flask's URL map without any explicit call.

## 🗺️ The Big Picture — Complete Auth Flow

Authentication in Flask-Login is a straightforward pipeline. Every request passes through it:

```text
REGISTRATION                       LOGIN
───────────                        ─────
User fills form                    User fills form
      │                                  │
      ▼                                  ▼
Validate & sanitize             Validate & sanitize
      │                                  │
      ▼                                  ▼
generate_password_hash()        Lookup user by email
      │                                  │
      ▼                                  ▼
Save User to DB              check_password_hash()
                                         │
                                         ▼
                                   login_user(user)
                                         │
                                         ▼
                              Session cookie set ──────────────────┐
                                                                    │
                                          Every subsequent request  │
                                                    │               │
                                                    ▼               │
                                         user_loader(user_id) ◄─────┘
                                                    │
                                                    ▼
                                            current_user proxy
                                                    │
                                                    ▼
                                      @login_required passes ✅
                                         or redirects ❌
                                                    │
                                         LOGOUT     │
                                                    ▼
                                           logout_user()
                                       (clears session cookie)
```

Flask-Login does **not** touch your database. It only manages the session cookie. You tell it
*how* to load a user by writing a `user_loader` callback — it calls that on every request.

> ❓ **Why are sessions needed if HTTP already sends cookies?** HTTP is stateless — each request is independent. A session uses a cookie to carry a small identifier (or signed data) so the server can recognise a returning user across multiple requests without requiring a login on every page load.

## 🧠 Core Concepts by Analogy

### 🎉 The Nightclub Wristband Analogy

Think of Flask-Login like a nightclub:

| Nightclub Step | Flask-Login Equivalent |
|---|---|
| You queue and show ID at the door | `POST /login` — user submits credentials |
| Bouncer checks your ID is real | `check_password_hash(stored, submitted)` |
| You get a wristband | `login_user(user)` — session cookie set |
| Inside, staff see your wristband | `current_user` — populated on every request |
| Wristband-only rooms | `@login_required` — bouncer at room door |
| You leave and hand back wristband | `logout_user()` — session cleared |

The wristband (session cookie) lets you move around freely without showing ID again.
Flask-Login is the bouncer system.

---

### 🔒 Password Hashing — The One-Way Lock

```text
    PASSWORD               HASH
    ────────               ────
    "secret123"   ──►  $2b$12$Xn3zK...  (bcrypt)
                           │
                    Stored in DB
                           │
                  ┌────────▼────────┐
                  │  No way back!   │
                  │  (one-way fn)   │
                  └────────────────┘

    At login time:
    "secret123" + stored_hash → check_password_hash() → True / False
```

Key insight: **the same password produces a different hash every time** (random salt).
That means even if two users have the same password, their stored hashes look completely
different — and rainbow-table attacks fail.

In [ ]:
# ── Minimal Flask-Login Setup ──────────────────────────────────────────
# Shows: LoginManager, UserMixin, user_loader, and the 4 UserMixin attributes

from flask import Flask
from flask_login import LoginManager, UserMixin, login_user, current_user

app = Flask(__name__)
app.secret_key = "demo-secret-key-change-in-prod"

login_manager = LoginManager()
login_manager.init_app(app)
login_manager.login_view = "login"          # redirect here if @login_required fails
login_manager.login_message = "Please log in to access this page."
login_manager.login_message_category = "info"

# ── Minimal in-memory "database" ─────────────────────────────────────────────
_users = {
    1: {"id": 1, "email": "alice@example.com", "name": "Alice"},
    2: {"id": 2, "email": "bob@example.com",   "name": "Bob"},
}

class User(UserMixin):
    """UserMixin provides: is_authenticated, is_active, is_anonymous, get_id()"""
    def __init__(self, data: dict):
        self.id    = data["id"]
        self.email = data["email"]
        self.name  = data["name"]

    def __repr__(self):
        return f"<User id={self.id} email={self.email!r}>"

@login_manager.user_loader
def load_user(user_id):
    """Called by Flask-Login on EVERY request to hydrate current_user."""
    data = _users.get(int(user_id))
    return User(data) if data else None   # returning None = AnonymousUser

# ── Demonstrate UserMixin attributes ─────────────────────────────────────────
alice = User(_users[1])

print("=== UserMixin attributes ===")
print(f"  is_authenticated : {alice.is_authenticated}")
print(f"  is_active        : {alice.is_active}")
print(f"  is_anonymous     : {alice.is_anonymous}")
print(f"  get_id()         : {alice.get_id()}")
print(f"  repr             : {alice!r}")

# Simulate loading from user_loader
loaded = load_user(1)
print(f"\nload_user(1)  → {loaded!r}")
print(f"load_user(99) → {load_user(99)!r}  ← returns None (missing user)")


In [ ]:
# ── Password Hashing Deep-Dive ───────────────────────────────────────────────
# werkzeug.security uses PBKDF2-HMAC-SHA256 by default (bcrypt via passlib optional)

from werkzeug.security import generate_password_hash, check_password_hash

password = "MyS3cur3P@ssword!"

hash1 = generate_password_hash(password)
hash2 = generate_password_hash(password)
hash3 = generate_password_hash(password, method="pbkdf2:sha256", salt_length=16)

print("=== Same password → different hash every time (random salt) ===")
print(f"  hash1 : {hash1}")
print(f"  hash2 : {hash2}")
print(f"  equal?: {hash1 == hash2}  ← always False")

print("\n=== check_password_hash ===")
print(f"  correct password : {check_password_hash(hash1, password)}")
print(f"  wrong password   : {check_password_hash(hash1, 'WrongPassword!')}")
print(f"  hash2 also works : {check_password_hash(hash2, password)}  ← same plaintext")

print("\n=== Hash anatomy ===")
parts = hash1.split("$")
print(f"  Full hash    : {hash1[:60]}...")
print(f"  Method prefix: {hash1.split(':')[0]}:{hash1.split(':')[1]}")
print(f"  Length       : {len(hash1)} chars  (safe to store in VARCHAR(256))")


In [ ]:
# ── Why Hashing Matters — Plain vs Hashed DB Dump ───────────────────────────

from werkzeug.security import generate_password_hash

print("=== PLAIN TEXT passwords in DB (NEVER do this) ===")
plain_db = [
    {"id": 1, "email": "alice@example.com", "password": "fluffy123"},
    {"id": 2, "email": "bob@example.com",   "password": "fluffy123"},
    {"id": 3, "email": "carol@example.com", "password": "dragon99"},
]
for row in plain_db:
    print(f"  {row}")

print()
print("  ☠️  If DB is leaked, ALL accounts are instantly compromised.")
print("  ☠️  Users sharing passwords → all their other accounts too.")

print()
print("=== HASHED passwords in DB (correct approach) ===")
hashed_db = [
    {"id": r["id"], "email": r["email"],
     "password_hash": generate_password_hash(r["password"])}
    for r in plain_db
]
for row in hashed_db:
    print(f"  id={row['id']} | {row['email']:<25} | {row['password_hash'][:45]}...")

print()
print("  ✅  Even if DB is leaked, attacker cannot recover plaintext.")
print("  ✅  Two users with same password have different hashes (salt).")

print()
print("=== Timing Attack Note ===")
print("  check_password_hash() uses hmac.compare_digest() internally.")
print("  This makes the comparison take the same time regardless of where")
print("  mismatches occur — preventing timing oracle attacks.")
print("  NEVER do: stored_hash == hash(submitted)  ← vulnerable to timing!")


In [ ]:
# ── Complete SQLAlchemy User Model ───────────────────────────────────────────
# Printed as a string so you can copy-paste into a real project.
# Combines: db.Model + UserMixin + password helpers

model_code = """
from flask_sqlalchemy import SQLAlchemy
from flask_login import UserMixin
from werkzeug.security import generate_password_hash, check_password_hash
from datetime import datetime

db = SQLAlchemy()

class User(UserMixin, db.Model):
    __tablename__ = "users"

    id         = db.Column(db.Integer, primary_key=True)
    username   = db.Column(db.String(64),  unique=True, nullable=False, index=True)
    email      = db.Column(db.String(120), unique=True, nullable=False, index=True)
    password_hash = db.Column(db.String(256), nullable=False)
    created_at = db.Column(db.DateTime, default=datetime.utcnow)
    is_active  = db.Column(db.Boolean, default=True)  # overrides UserMixin default

    # Relationships (add as needed)
    # posts = db.relationship("Post", backref="author", lazy="dynamic")

    def set_password(self, password: str) -> None:
        """Hash and store the password. Never store plaintext."""
        self.password_hash = generate_password_hash(password)

    def check_password(self, password: str) -> bool:
        """Return True if password matches the stored hash."""
        return check_password_hash(self.password_hash, password)

    def __repr__(self):
        return f"<User {self.username!r}>"
"""
print(model_code)


In [ ]:
# ── Registration Route ───────────────────────────────────────────────────────

registration_route = """
from flask import render_template, redirect, url_for, flash
from flask_login import login_user, current_user

@app.route("/register", methods=["GET", "POST"])
def register():
    if current_user.is_authenticated:
        return redirect(url_for("dashboard"))  # already logged in

    form = RegistrationForm()
    if form.validate_on_submit():
        user = User(
            username=form.username.data,
            email=form.email.data.lower(),
        )
        user.set_password(form.password.data)  # ← hashes the password
        db.session.add(user)
        db.session.commit()

        flash("Account created! Please log in.", "success")
        return redirect(url_for("login"))

    return render_template("register.html", form=form)
"""
print(registration_route)


In [ ]:
# ── Registration Form with Custom Validators ─────────────────────────────────
# Uses WTForms + Flask-WTF. Demonstrates custom validate_<field> methods.

registration_form_code = """
from flask_wtf import FlaskForm
from wtforms import StringField, PasswordField, SubmitField
from wtforms.validators import DataRequired, Email, Length, EqualTo, ValidationError
from .models import User

class RegistrationForm(FlaskForm):
    username = StringField(
        "Username",
        validators=[DataRequired(), Length(min=3, max=64)]
    )
    email = StringField(
        "Email",
        validators=[DataRequired(), Email(), Length(max=120)]
    )
    password = PasswordField(
        "Password",
        validators=[DataRequired(), Length(min=8, message="Minimum 8 characters")]
    )
    confirm_password = PasswordField(
        "Confirm Password",
        validators=[DataRequired(), EqualTo("password", message="Passwords must match")]
    )
    submit = SubmitField("Create Account")

    def validate_username(self, field):
        """Custom validator: called automatically by WTForms."""
        if User.query.filter_by(username=field.data).first():
            raise ValidationError("Username already taken. Please choose another.")

    def validate_email(self, field):
        """Custom validator: block duplicate emails."""
        if User.query.filter_by(email=field.data.lower()).first():
            raise ValidationError("An account with that email already exists.")
"""
print(registration_form_code)

print("--- Key validator notes ---")
print("  EqualTo('password') → ensures confirm_password matches password field")
print("  validate_<fieldname> → WTForms calls these automatically on form.validate()")
print("  Both custom validators query the DB to enforce uniqueness at form level")


In [ ]:
# ── Login Route — login_user + next_page handling ────────────────────────────

login_route_code = """
from flask import render_template, redirect, url_for, flash, request
from flask_login import login_user, current_user
from urllib.parse import urlparse

@app.route("/login", methods=["GET", "POST"])
def login():
    if current_user.is_authenticated:
        return redirect(url_for("dashboard"))

    form = LoginForm()
    if form.validate_on_submit():
        user = User.query.filter_by(email=form.email.data.lower()).first()

        if user is None or not user.check_password(form.password.data):
            flash("Invalid email or password.", "danger")
            return redirect(url_for("login"))

        login_user(user, remember=form.remember_me.data)

        # Safe redirect: only follow ?next= if it points to same host
        next_page = request.args.get("next")
        if not next_page or urlparse(next_page).netloc != "":
            next_page = url_for("dashboard")

        flash(f"Welcome back, {user.username}!", "success")
        return redirect(next_page)

    return render_template("login.html", form=form)
"""
print(login_route_code)

print("--- Why validate the ?next= parameter? ---")
print("  Without validation, an attacker could craft:")
print("  /login?next=https://evil.com  → open redirect vulnerability")
print("  urlparse(next_page).netloc != '' catches absolute URLs to other hosts.")


In [ ]:
# ── Login Form with Remember Me ──────────────────────────────────────────────

login_form_code = """
from flask_wtf import FlaskForm
from wtforms import StringField, PasswordField, BooleanField, SubmitField
from wtforms.validators import DataRequired, Email

class LoginForm(FlaskForm):
    email = StringField(
        "Email",
        validators=[DataRequired(), Email()]
    )
    password = PasswordField(
        "Password",
        validators=[DataRequired()]
    )
    remember_me = BooleanField("Remember Me")   # ← maps to login_user(remember=...)
    submit = SubmitField("Log In")
"""
print(login_form_code)

print("--- BooleanField notes ---")
print("  Renders as <input type='checkbox'>")
print("  form.remember_me.data → True / False")
print("  Passed directly to login_user(user, remember=form.remember_me.data)")


In [ ]:
# ── @login_required Decorator + logout_user ──────────────────────────────────

from flask import Flask, redirect, url_for, jsonify
from flask_login import (LoginManager, UserMixin, login_user,
                          logout_user, login_required, current_user)

app2 = Flask(__name__)
app2.secret_key = "demo-secret-2"

lm = LoginManager(app2)
lm.login_view = "login_page"

class DemoUser(UserMixin):
    def __init__(self, uid):
        self.id = uid

@lm.user_loader
def load_demo(uid):
    return DemoUser(uid)

@app2.route("/dashboard")
@login_required            # ← Flask-Login checks current_user.is_authenticated
def dashboard():           #   If False → redirect to login_manager.login_view
    return jsonify({"msg": f"Hello user {current_user.id}!"})

@app2.route("/logout")
@login_required
def logout_view():
    logout_user()          # ← clears the session cookie
    return jsonify({"msg": "Logged out"})

# ── Demonstrate with test client ─────────────────────────────────────────────
with app2.test_client() as c:
    # 1. Hit dashboard without logging in
    resp = c.get("/dashboard")
    print(f"Unauthenticated /dashboard → {resp.status_code} (302 redirect to login)")
    print(f"  Location: {resp.headers.get('Location', '')}")

    # 2. Simulate login then access dashboard
    with app2.test_request_context():
        with c.session_transaction() as sess:
            # Flask-Login stores user_id in session under "_user_id"
            sess["_user_id"] = "42"

    resp2 = c.get("/dashboard")
    print(f"\nAuthenticated /dashboard → {resp2.status_code}")
    print(f"  Response: {resp2.get_json()}")

    # 3. Logout
    resp3 = c.get("/logout")
    print(f"\n/logout → {resp3.status_code}")
    print(f"  Response: {resp3.get_json()}")


In [ ]:
# ── current_user in Python Views — Live Demo ─────────────────────────────────
# current_user is a thread-local proxy; it changes per request.

from flask import Flask, jsonify
from flask_login import LoginManager, UserMixin, current_user

app3 = Flask(__name__)
app3.secret_key = "demo-secret-3"
lm3 = LoginManager(app3)

class BlogUser(UserMixin):
    def __init__(self, uid, name, role="reader"):
        self.id   = uid
        self.name = name
        self.role = role

_db = {1: BlogUser(1, "Alice", "admin"), 2: BlogUser(2, "Bob", "reader")}

@lm3.user_loader
def load3(uid):
    return _db.get(int(uid))

@app3.route("/profile")
def profile():
    if current_user.is_authenticated:
        return jsonify({
            "authenticated": True,
            "id":   current_user.id,
            "name": current_user.name,
            "role": current_user.role,
        })
    return jsonify({"authenticated": False, "user": "anonymous"})

# ── Test: anonymous request ───────────────────────────────────────────────────
with app3.test_client() as c:
    resp = c.get("/profile")
    print("=== Anonymous request ===")
    print(" ", resp.get_json())

    # Authenticate as Alice
    with c.session_transaction() as sess:
        sess["_user_id"] = "1"

    resp2 = c.get("/profile")
    print("\n=== Authenticated as Alice ===")
    print(" ", resp2.get_json())

    # Switch to Bob
    with c.session_transaction() as sess:
        sess["_user_id"] = "2"

    resp3 = c.get("/profile")
    print("\n=== Authenticated as Bob ===")
    print(" ", resp3.get_json())

print("\ncurrent_user automatically changes per request context —")
print("no global state, no threading issues.")


In [ ]:
# ── current_user in Jinja2 Templates ─────────────────────────────────────────
# Flask-Login injects current_user into every template context automatically.
# No need to pass it manually in render_template().

template_snippets = """
{# ── base.html — nav bar ── #}
<nav>
  {% if current_user.is_authenticated %}
    <span>Hello, {{ current_user.username }}!</span>
    <a href="{{ url_for('logout') }}">Log Out</a>

    {% if current_user.role == 'admin' %}
      <a href="{{ url_for('admin_panel') }}">Admin</a>
    {% endif %}

  {% else %}
    <a href="{{ url_for('login') }}">Log In</a>
    <a href="{{ url_for('register') }}">Register</a>
  {% endif %}
</nav>

{# ── post.html — show Edit button only to post author ── #}
{% if current_user.is_authenticated and current_user.id == post.author_id %}
  <a href="{{ url_for('edit_post', post_id=post.id) }}">Edit</a>
  <a href="{{ url_for('delete_post', post_id=post.id) }}">Delete</a>
{% endif %}

{# ── Accessing any attribute your User model has ── #}
<p>Member since: {{ current_user.created_at.strftime('%B %Y') }}</p>
"""

print(template_snippets)
print("Key points:")
print("  - current_user is available in ALL templates (injected by Flask-Login)")
print("  - current_user.is_authenticated → check before accessing user attributes")
print("  - For anonymous users, current_user.is_anonymous == True")
print("  - You can access ANY attribute defined on your User model")


In [ ]:
# ── LoginManager Configuration Options ───────────────────────────────────────

from flask import Flask
from flask_login import LoginManager

app4 = Flask(__name__)
app4.secret_key = "demo"

login_manager = LoginManager()
login_manager.init_app(app4)

# ── The most important settings ───────────────────────────────────────────────
login_manager.login_view          = "auth.login"      # blueprint.endpoint or plain endpoint
login_manager.login_message       = "Please sign in to continue."
login_manager.login_message_category = "warning"      # Bootstrap alert class

# ── How the ?next= parameter works ───────────────────────────────────────────
print("=== ?next= parameter flow ===")
print("  1. User visits /dashboard (protected)")
print("  2. @login_required → redirect to /login?next=%2Fdashboard")
print("  3. User logs in successfully")
print("  4. Login route reads request.args.get('next') → '/dashboard'")
print("  5. Redirect back to /dashboard — seamless UX")
print()

# Demonstrate with test client
@app4.route("/secret")
def secret():
    from flask_login import login_required
    return "secret content"

with app4.test_client() as c:
    resp = c.get("/secret")
    location = resp.headers.get("Location", "")
    print(f"GET /secret (unauthenticated) → {resp.status_code}")
    print(f"  Redirects to: {location}")
    if "next" in location:
        print("  ✅ ?next= parameter is set automatically by Flask-Login")


In [ ]:
# ── Remember Me — Persistent Login Cookie ────────────────────────────────────
# login_user(remember=True) sets a long-lived cookie separate from the session cookie.

from flask import Flask
from flask_login import LoginManager, UserMixin, login_user
from datetime import timedelta

app5 = Flask(__name__)
app5.secret_key = "demo-secret-5"

# ── Configuration ─────────────────────────────────────────────────────────────
app5.config["REMEMBER_COOKIE_DURATION"]  = timedelta(days=30)
app5.config["REMEMBER_COOKIE_SECURE"]    = True   # HTTPS only in production
app5.config["REMEMBER_COOKIE_HTTPONLY"]  = True   # not accessible via JS
app5.config["REMEMBER_COOKIE_SAMESITE"]  = "Lax"  # CSRF protection

lm5 = LoginManager(app5)

class SimpleUser(UserMixin):
    def __init__(self, uid):
        self.id = uid

@lm5.user_loader
def load5(uid):
    return SimpleUser(uid)

print("=== Remember Me: what happens ===")
print()
print("  login_user(user, remember=False)  ← default")
print("  → Browser session cookie only")
print("  → Cleared when browser is closed")
print()
print("  login_user(user, remember=True)")
print("  → Session cookie  +  'remember_token' cookie")
print("  → 'remember_token' persists for REMEMBER_COOKIE_DURATION (30 days)")
print("  → User stays logged in after browser restart")
print()
print("  Cookie security flags (always set in production):")
print("  ┌─────────────────────┬─────────────────────────────────────────────┐")
print("  │ REMEMBER_COOKIE_SECURE   │ Send only over HTTPS                   │")
print("  │ REMEMBER_COOKIE_HTTPONLY │ Block JS access (XSS mitigation)        │")
print("  │ REMEMBER_COOKIE_SAMESITE │ Block cross-site sending (CSRF defense) │")
print("  └─────────────────────┴─────────────────────────────────────────────┘")


## ⚖️ Comparisons

### Flask-Login vs Manual Session Auth

| Feature | Manual `session[]` | Flask-Login |
|---|---|---|
| User loading per request | Write it yourself | `@user_loader` callback |
| `current_user` proxy | None — pass explicitly | Built-in, available everywhere |
| `@login_required` decorator | Write it yourself | Built-in |
| Remember Me cookie | Write it yourself | `login_user(remember=True)` |
| Anonymous user handling | None — check manually | `AnonymousUser` with safe defaults |
| Logout (clear session) | `session.clear()` | `logout_user()` |
### Password Storage Methods

| Method | Secure? | Notes |
|---|---|---|
| Plain text | ❌ Never | Instant breach |
| MD5 / SHA1 | ❌ No | Broken, rainbow tables trivial |
| SHA256 (no salt) | ❌ No | Rainbow tables work |
| bcrypt / PBKDF2 with salt | ✅ Yes | Werkzeug default |
| Argon2 | ✅ Best | Use `flask-bcrypt` or `argon2-cffi` |

In [ ]:
# ── Manual Session Auth vs Flask-Login — Boilerplate Comparison ─────────────

manual_approach = """
# ─── MANUAL SESSION APPROACH ─────────────────────────────────────────────────
# Every protected route needs this boilerplate:

@app.route("/dashboard")
def dashboard():
    if "user_id" not in session:
        flash("Please log in.", "warning")
        return redirect(url_for("login"))
    user = User.query.get(session["user_id"])
    if user is None:
        session.clear()
        return redirect(url_for("login"))
    # ... 6 lines of boilerplate before actual logic
    return render_template("dashboard.html", user=user)

# Logout:
@app.route("/logout")
def logout():
    session.pop("user_id", None)
    return redirect(url_for("login"))
"""

flask_login_approach = """
# ─── FLASK-LOGIN APPROACH ─────────────────────────────────────────────────────
# One decorator. current_user just works.

@app.route("/dashboard")
@login_required              # ← handles ALL the boilerplate above
def dashboard():
    return render_template("dashboard.html")   # current_user in template context

# Logout:
@app.route("/logout")
@login_required
def logout():
    logout_user()
    return redirect(url_for("login"))
"""

print("MANUAL SESSION APPROACH:")
print(manual_approach)
print("FLASK-LOGIN APPROACH:")
print(flask_login_approach)

# Count lines of boilerplate
manual_lines = len([l for l in manual_approach.strip().splitlines() if l.strip() and not l.strip().startswith("#")])
fl_lines = len([l for l in flask_login_approach.strip().splitlines() if l.strip() and not l.strip().startswith("#")])
print(f"Lines of logic code — Manual: {manual_lines}  |  Flask-Login: {fl_lines}")
print("Flask-Login eliminates repeated boilerplate and centralises auth logic.")


In [ ]:
# ── Plain Text vs Hashed — Mock DB Dump + Timing Attack Demo ────────────────

import time
import hmac
from werkzeug.security import generate_password_hash, check_password_hash

# Mock DB
users_plain  = [("alice", "fluffy123"), ("bob", "fluffy123"), ("carol", "dragon99")]
users_hashed = [(u, generate_password_hash(p)) for u, p in users_plain]

print("=== If DB is leaked ===")
print("PLAIN TEXT dump:")
for u, p in users_plain:
    print(f"  {u:<8} | {p}")
print("→ Every password visible. Pattern visible: alice and bob share a password.")

print()
print("HASHED dump:")
for u, h in users_hashed:
    print(f"  {u:<8} | {h[:55]}...")
print("→ No password visible. alice and bob hashes look completely different.")

print()
print("=== Timing Attack Demo ===")
print("Naive comparison (vulnerable):")
stored = generate_password_hash("correct")
attempts = ["correct", "wrong", "x"]
for attempt in attempts:
    start = time.perf_counter_ns()
    # Naive — short-circuits on first mismatch
    result = (stored == generate_password_hash(attempt))
    elapsed = time.perf_counter_ns() - start
    print(f"  {attempt!r:<10} naive: {elapsed:>8} ns")

print()
print("check_password_hash (safe — constant time comparison):")
for attempt in attempts:
    start = time.perf_counter_ns()
    result = check_password_hash(stored, attempt)
    elapsed = time.perf_counter_ns() - start
    print(f"  {attempt!r:<10} safe:  {elapsed:>8} ns  result={result}")


In [ ]:
# ── What If #1: Forgetting @login_required ───────────────────────────────────
# Without the decorator, ANY visitor (anonymous or not) can access the route.

from flask import Flask, jsonify
from flask_login import LoginManager, UserMixin, current_user

app6 = Flask(__name__)
app6.secret_key = "demo6"
lm6 = LoginManager(app6)

class U(UserMixin):
    def __init__(self, uid): self.id = uid

@lm6.user_loader
def load6(uid): return U(uid)

# ────────────────────────────────────────────────────────────────────────────
# INSECURE: no @login_required
@app6.route("/admin/delete-all")
def delete_all_insecure():
    # This runs for ANYONE — authenticated or not
    return jsonify({"action": "deleted everything", "user": str(current_user)})

# SECURE: with @login_required
from flask_login import login_required

@app6.route("/admin/delete-all-safe")
@login_required
def delete_all_secure():
    return jsonify({"action": "deleted everything", "user": str(current_user)})

# ── Demonstrate the hole ──────────────────────────────────────────────────────
with app6.test_client() as c:
    print("=== Hitting INSECURE route as anonymous user ===")
    resp = c.get("/admin/delete-all")
    print(f"  Status : {resp.status_code}  ← 200! No protection.")
    print(f"  Body   : {resp.get_json()}")
    print("  ☠️  Anonymous user just 'deleted everything'!")

    print()
    print("=== Hitting SECURE route as anonymous user ===")
    resp2 = c.get("/admin/delete-all-safe")
    print(f"  Status : {resp2.status_code}  ← 302 redirect to login ✅")
    print(f"  Location: {resp2.headers.get('Location', '')}")

print()
print("Rule: Every route that touches user-specific or sensitive data needs @login_required.")
print("Use it liberally — it costs nothing and prevents major security holes.")


In [ ]:
# ── What If #2: user_loader Returns None → AnonymousUser ────────────────────
# Flask-Login NEVER crashes when user_loader returns None.
# It silently falls back to an AnonymousUser object.

from flask import Flask, jsonify
from flask_login import (LoginManager, UserMixin, current_user,
                          AnonymousUserMixin, login_required)

app7 = Flask(__name__)
app7.secret_key = "demo7"
lm7 = LoginManager(app7)
lm7.login_view = "login"

@lm7.user_loader
def load7(uid):
    print(f"  [user_loader] called with uid={uid!r}")
    return None   # ← simulate: user not found in DB (deleted? bad session?)

@app7.route("/whoami")
def whoami():
    return jsonify({
        "is_authenticated": current_user.is_authenticated,
        "is_anonymous":     current_user.is_anonymous,
        "type":             type(current_user).__name__,
    })

@app7.route("/protected")
@login_required
def protected():
    return jsonify({"msg": "you are in"})

# ── Test ──────────────────────────────────────────────────────────────────────
with app7.test_client() as c:
    print("=== Session has user_id but user_loader returns None ===")
    with c.session_transaction() as sess:
        sess["_user_id"] = "999"   # simulate stale session for deleted user

    resp = c.get("/whoami")
    print(f"  /whoami → {resp.get_json()}")
    print()
    resp2 = c.get("/protected")
    print(f"  /protected → status {resp2.status_code} (redirected to login, not crashed)")

print()
print("AnonymousUserMixin attributes:")
anon = AnonymousUserMixin()
print(f"  is_authenticated : {anon.is_authenticated}")
print(f"  is_anonymous     : {anon.is_anonymous}")
print(f"  is_active        : {anon.is_active}")
print(f"  get_id()         : {anon.get_id()}")
print()
print("→ Templates and routes can safely call current_user.is_authenticated")
print("  without any None checks — AnonymousUser returns safe defaults.")


In [ ]:
# ── What If #3: User Deleted While Logged In ─────────────────────────────────
# Scenario: Alice is logged in. Admin deletes her account mid-session.
# What happens on Alice's next request?

from flask import Flask, jsonify
from flask_login import LoginManager, UserMixin, login_required, current_user

app8 = Flask(__name__)
app8.secret_key = "demo8"
lm8 = LoginManager(app8)
lm8.login_view = "login"

# Mutable "database" — we'll delete Alice during the demo
active_users = {1: {"id": 1, "name": "Alice"}}

class AppUser(UserMixin):
    def __init__(self, data):
        self.id   = data["id"]
        self.name = data["name"]

@lm8.user_loader
def load8(uid):
    data = active_users.get(int(uid))
    print(f"  [user_loader] uid={uid} → {'found' if data else 'NOT FOUND (returns None)'}")
    return AppUser(data) if data else None

@app8.route("/dashboard")
@login_required
def dashboard():
    return jsonify({"msg": f"Hello {current_user.name}"})

# ── Trace the flow ────────────────────────────────────────────────────────────
with app8.test_client() as c:
    # Alice is logged in
    with c.session_transaction() as sess:
        sess["_user_id"] = "1"

    print("=== Request 1: Alice is in DB — normal access ===")
    resp = c.get("/dashboard")
    print(f"  Status: {resp.status_code}  Body: {resp.get_json()}")

    # Admin deletes Alice from DB
    print()
    print("=== Admin action: deleting Alice from database ===")
    del active_users[1]
    print("  active_users:", active_users)

    print()
    print("=== Request 2: Alice's session cookie is still valid ===")
    resp2 = c.get("/dashboard")
    print(f"  Status: {resp2.status_code}  Location: {resp2.headers.get('Location', '')}")

print()
print("Trace:")
print("  1. Alice's browser still has session cookie with user_id=1")
print("  2. Flask-Login calls user_loader(1)")
print("  3. user_loader returns None (user deleted from DB)")
print("  4. Flask-Login sets current_user = AnonymousUser")
print("  5. @login_required sees is_authenticated=False → redirect to login")
print("  6. Alice is effectively force-logged-out without any special code!")
print()
print("  ✅ No additional code needed — Flask-Login handles this automatically.")


## 🔗 Real-World Project: Flask Blog with Auth

A complete working example of this chapter's concepts:

```text
flask_blog/
├── app/
│   ├── __init__.py          # LoginManager.init_app(app)
│   ├── models.py            # User model with UserMixin + set_password/check_password
│   ├── auth/
│   │   ├── __init__.py      # Blueprint: url_prefix="/auth"
│   │   ├── routes.py        # /register, /login, /logout
│   │   └── forms.py         # RegistrationForm, LoginForm
│   └── templates/
│       ├── base.html        # current_user in nav
│       ├── auth/
│       │   ├── login.html
│       │   └── register.html
│       └── dashboard.html   # @login_required protected
├── config.py                # SECRET_KEY, REMEMBER_COOKIE_DURATION
└── run.py
```

Key files pattern:
```python
# app/__init__.py
from flask_login import LoginManager
login_manager = LoginManager()

def create_app(config):
    app = Flask(__name__)
    app.config.from_object(config)
    login_manager.init_app(app)
    login_manager.login_view = "auth.login"

    from .auth import bp as auth_bp
    app.register_blueprint(auth_bp)
    return app
```

> ❓ **Why split code into Blueprints?** A single file with hundreds of routes becomes unreadable. Blueprints group related routes (e.g., all auth routes) into pluggable mini-applications that you register on the main app — like separate chapters in a book.

In [ ]:
# ── Chapter 10 Cheat Sheet ───────────────────────────────────────────────────

cheat_sheet = """
╔══════════════════════════════════════════════════════════════════════════════╗
║          FLASK-LOGIN CHEAT SHEET — Chapter 10                              ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  SETUP                                                                      ║
║  ──────────────────────────────────────────────────────────────────────    ║
║  login_manager = LoginManager(app)                                          ║
║  login_manager.login_view = "auth.login"                                    ║
║                                                                             ║
║  @login_manager.user_loader                                                 ║
║  def load_user(user_id):                                                    ║
║      return User.query.get(int(user_id))  # return None if not found        ║
║                                                                             ║
║  USER MODEL                                                                 ║
║  ──────────────────────────────────────────────────────────────────────    ║
║  class User(UserMixin, db.Model): ...    # UserMixin provides 4 properties ║
║  UserMixin gives: is_authenticated, is_active, is_anonymous, get_id()      ║
║                                                                             ║
║  PASSWORDS                                                                  ║
║  ──────────────────────────────────────────────────────────────────────    ║
║  generate_password_hash(password)       # hash (random salt each time)     ║
║  check_password_hash(stored, submitted) # True / False                      ║
║                                                                             ║
║  AUTH ACTIONS                                                               ║
║  ──────────────────────────────────────────────────────────────────────    ║
║  login_user(user)                       # set session cookie               ║
║  login_user(user, remember=True)        # + persistent remember cookie     ║
║  logout_user()                          # clear session                    ║
║  @login_required                        # protect a view                   ║
║                                                                             ║
║  CURRENT USER                                                               ║
║  ──────────────────────────────────────────────────────────────────────    ║
║  current_user.is_authenticated          # True if logged in                ║
║  current_user.is_anonymous              # True if not logged in            ║
║  current_user.<any_model_attr>          # access User model attributes     ║
║  Available in templates automatically — no need to pass manually           ║
║                                                                             ║
║  REMEMBER ME CONFIG                                                         ║
║  ──────────────────────────────────────────────────────────────────────    ║
║  REMEMBER_COOKIE_DURATION  = timedelta(days=30)                            ║
║  REMEMBER_COOKIE_SECURE    = True    # HTTPS only                          ║
║  REMEMBER_COOKIE_HTTPONLY  = True    # no JS access                        ║
║  REMEMBER_COOKIE_SAMESITE  = "Lax"  # CSRF defense                        ║
║                                                                             ║
║  EDGE CASES                                                                 ║
║  ──────────────────────────────────────────────────────────────────────    ║
║  user_loader returns None → AnonymousUser (no crash)                       ║
║  User deleted mid-session → next request = AnonymousUser = redirect        ║
║  ?next= param → validate netloc to prevent open redirect attacks           ║
╚══════════════════════════════════════════════════════════════════════════════╝
"""
print(cheat_sheet)


## 🏋️ Practice Prompts

Test your understanding by building these:

1. **Basic auth** — Create a Flask app with `/register` and `/login` routes, a `User` model with `set_password`/`check_password`, and a `/dashboard` protected by `@login_required`.

2. **Profile page** — Add a `/profile/<username>` route. Allow any visitor to *view* a profile, but only the logged-in owner can *edit* it. Use `current_user.id == user.id` check.

3. **Remember Me** — Add a "Remember Me" checkbox to your login form. Set `REMEMBER_COOKIE_DURATION` to 7 days. Verify the cookie persists in browser DevTools.

4. **Email-based login** — Modify the login form to use email instead of username. Normalize with `.lower()` before querying.

5. **Role-based access** — Add a `role` column (`"user"` / `"admin"`) to `User`. Create an `admin_required` decorator (similar to `@login_required`) that also checks `current_user.role == "admin"`.

6. **"What If" challenge** — Deliberately remove `@login_required` from one of your protected routes. Write a test that demonstrates an anonymous request returns 200. Then add the decorator back and show the test now gets a 302.

7. **Force logout** — Write a `/admin/force-logout/<user_id>` route that "deletes" a user from your in-memory store. Demonstrate that a test client with that user's session gets redirected on the next request.

<div style='width:100%; display:flex; justify-content:space-between; align-items:center; margin: 1em 0;'>
  <a href='../3.%20application_structure/9.%20configuration_management.ipynb' style='padding:6px 14px; background:#007bff; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>← Chapter 9: Configuration</a>
  <a href='../TOC.md' style='padding:6px 14px; background:#6c757d; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>📚 Table of Contents</a>
  <a href='11.%20security_best_practices.ipynb' style='padding:6px 14px; background:#28a745; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>Chapter 11: Security →</a>
</div>

<div style='width:100%; display:flex; justify-content:space-between; align-items:center; margin: 1em 0;'>
  <a href='../3. application_structure/9. configuration_management.ipynb' style='font-weight:bold; font-size:1.05em;'>&larr; Previous</a>
  <a href='../TOC.md' style='font-weight:bold; font-size:1.05em; text-align:center;'>Table of Contents</a>
  <a href='11. security_best_practices.ipynb' style='font-weight:bold; font-size:1.05em;'>Next &rarr;</a>
</div>